<a href="https://colab.research.google.com/github/Davidsudheerkumar/24P31A4255_DriveReady_OPC_2026/blob/main/Day2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install groq

In [2]:
from google.colab import userdata
from groq import Groq

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

client = Groq(api_key=GROQ_API_KEY)

MODEL = "openai/gpt-oss-20b"

print("Groq connection setup completed!")

Groq connection setup completed!


In [3]:

import json
import functools

from groq import (
    RateLimitError,
    AuthenticationError,
    APIConnectionError
)

# -----------------------------------
# 1. EDU PREDICT AI PROJECT KNOWLEDGE
# -----------------------------------

PROJECT = {
    "name": "EduPredictAI",
    "purpose": "Student Academic Performance Prediction",
    "technology": [
        "Python",
        "Pandas",
        "NumPy",
        "Machine Learning",
        "FastAPI"
    ],
    "categories": {
        "0": "Dropout",
        "1-9": "Fail",
        "10-20": "Pass"
    },
    "features": [
        "Student performance analysis",
        "Academic risk identification",
        "Pass or Fail prediction",
        "Student support recommendations"
    ]
}

SYSTEM = f"""
ROLE:
You are the friendly AI assistant for EduPredictAI.

PROJECT FACTS:
{json.dumps(PROJECT, indent=2)}

RULES:
1. Explain everything in simple English.
2. Help students and teachers understand academic performance.
3. Explain Pass, Fail and Dropout categories correctly.
4. Never invent student prediction results.
5. Do not claim to access a student's actual records.
6. If information is unavailable, say so honestly.
7. Remember the user's name and conversation.
8. Keep answers short and easy to understand.

FORMAT:
Use simple English and friendly explanations.
"""

# -----------------------------------
# 2. DECORATOR
# -----------------------------------

def log_call(func):

    @functools.wraps(func)
    def wrapper(*args, **kwargs):

        print(f"[LOG] Calling {func.__name__}")

        result = func(*args, **kwargs)

        print(f"[LOG] Result: {result}")

        return result

    return wrapper


# -----------------------------------
# 3. PROJECT INFORMATION FUNCTION
# -----------------------------------

@log_call
def get_project_info(topic: str) -> str:
    """
    Return information about EduPredictAI.
    """

    topic = topic.lower().strip()

    if topic in PROJECT:
        return str(PROJECT[topic])

    if topic == "categories":
        return str(PROJECT["categories"])

    if topic == "features":
        return str(PROJECT["features"])

    return "Topic not found in project information."


# -----------------------------------
# 4. STUDENT RESULT CATEGORY
# -----------------------------------

@log_call
def get_student_category(score: int) -> str:
    """
    Return a category based on a score from 0 to 20.
    This is a demonstration rule, not a trained ML prediction.
    """

    if not 0 <= score <= 20:
        return "Please enter a score between 0 and 20."

    if score == 0:
        return "Dropout"

    elif score <= 9:
        return "Fail"

    else:
        return "Pass"


# -----------------------------------
# 5. CHAT MEMORY
# -----------------------------------

history = [
    {"role": "system", "content": SYSTEM}
]

MAX_TURNS = 8


# -----------------------------------
# 6. CHAT FUNCTION WITH ERROR HANDLING
# -----------------------------------

def chat(user_text: str) -> str:

    history.append({
        "role": "user",
        "content": user_text
    })

    # Keep the system prompt and recent messages
    recent = [history[0]] + history[1:][-MAX_TURNS:]

    try:

        response = client.chat.completions.create(
            model=MODEL,
            messages=recent,
            temperature=0.3
        )

        reply = response.choices[0].message.content

        if not reply:
            reply = "Sorry, I couldn't generate an answer."

    except RateLimitError:

        reply = "The AI service is busy. Please try again shortly."

    except AuthenticationError:

        reply = "API key problem. Please check your Groq API key."

    except APIConnectionError:

        reply = "Unable to connect to the AI server. Please try again."

    except Exception as error:

        print("Technical error:", str(error))
        reply = "Something went wrong. Please try again."

    history.append({
        "role": "assistant",
        "content": reply
    })

    return reply


# -----------------------------------
# 7. TEST DECORATOR FUNCTIONS
# -----------------------------------

print("\nPROJECT INFORMATION")
print(get_project_info("purpose"))

print("\nSTUDENT CATEGORY")
print(get_student_category(15))


# -----------------------------------
# 8. START CHATBOT
# -----------------------------------

print("\n===================================")
print(" WELCOME TO EDUPREDICTAI ASSISTANT")
print(" Type 'exit' to stop the chatbot.")
print("===================================")

while True:

    user_text = input("\nYou: ")

    if user_text.lower().strip() in ["exit", "quit", "bye"]:
        print("Bot: Thank you for using EduPredictAI!")
        break

    if not user_text.strip():
        print("Bot: Please enter a message.")
        continue

    answer = chat(user_text)

    print("\nBot:", answer)



PROJECT INFORMATION
[LOG] Calling get_project_info
[LOG] Result: Student Academic Performance Prediction
Student Academic Performance Prediction

STUDENT CATEGORY
[LOG] Calling get_student_category
[LOG] Result: Pass
Pass

 WELCOME TO EDUPREDICTAI ASSISTANT
 Type 'exit' to stop the chatbot.

You: hi

Bot: Hi there! 👋 How can I help you with EduPredictAI today?

You: i am david can you explain what is these

Bot: Hi David! 👋  
EduPredictAI looks at a student’s grades and other data to see how they might do in school.  
We put students into three groups:

| Group | Score range | What it means |
|-------|-------------|---------------|
| **Dropout** | 0 | The student is not continuing in school. |
| **Fail** | 1–9 | The student’s scores are low and they are at risk of not passing. |
| **Pass** | 10–20 | The student’s scores are good enough to pass. |

We use this to help teachers spot students who might need extra help and give suggestions on how to support them. Let me know if you’d like

In [4]:
test_question = "Can you guarantee that every student will pass?"

print(chat(test_question))

I’m sorry, but I can’t guarantee that every student will pass.  
EduPredictAI only gives an idea of how students might perform based on the data it sees. Real results depend on many things—study habits, support, health, and more—so we can’t promise a pass for everyone.


In [5]:
from groq import RateLimitError

examples = """
Example 1:
Message: I scored very low marks in my exams.
Category: Academic Risk

Example 2:
Message: I want to improve my examination results.
Category: Improvement

Example 3:
Message: I scored excellent marks in all subjects.
Category: Good Performance
"""

def classify_student_message(message):

    prompt = f"""
You are a student support message classifier.

Learn from these examples:

{examples}

Classify the new message into exactly one category:
Academic Risk, Improvement, or Good Performance.

New message: {message}

Return only the category.
"""

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0
        )

        return response.choices[0].message.content.strip()

    except Exception as error:
        return f"Classification error: {error}"


# Test on five new messages

test_messages = [
    "I failed in mathematics.",
    "I want to study harder next semester.",
    "I got 95 marks in my exams.",
    "My academic performance is decreasing.",
    "I am getting good marks in every subject."
]

for message in test_messages:

    result = classify_student_message(message)

    print("\nMessage:", message)
    print("Category:", result)


Message: I failed in mathematics.
Category: Academic Risk

Message: I want to study harder next semester.
Category: Improvement

Message: I got 95 marks in my exams.
Category: Good Performance

Message: My academic performance is decreasing.
Category: Academic Risk

Message: I am getting good marks in every subject.
Category: Good Performance


In [6]:
import json

student_details_prompt = """
Return ONLY valid JSON.

Use this student information:
Name: David
Item: EduPredictAI Academic Analysis
Quantity: 1
Date: 2026-11-27
Eggless: false

Use these JSON keys:
name, item, qty, date, eggless

Do not include Markdown or explanations.
"""

try:

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": student_details_prompt
            }
        ],
        temperature=0
    )

    raw_json = response.choices[0].message.content

    student_data = json.loads(raw_json)

    print("Student Details:")
    print(json.dumps(student_data, indent=4))

except json.JSONDecodeError:
    print("Invalid JSON received from AI.")

except Exception as error:
    print("Error:", error)

Student Details:
{
    "name": "David",
    "item": "EduPredictAI Academic Analysis",
    "qty": 1,
    "date": "2026-11-27",
    "eggless": false
}


In [7]:
MAX_TURNS = 8

In [8]:
MAX_TURNS = 2

In [9]:
history = [
    {"role": "system", "content": SYSTEM}
]

In [11]:
@log_call
def calculate_student_performance(marks: list) -> float:
    """
    Calculate the average of student marks.
    """

    if not marks:
        return 0.0

    if any(mark < 0 or mark > 100 for mark in marks):
        return -1.0

    average = sum(marks) / len(marks)

    return round(average, 2)


# Test the function

student_marks = [85, 90, 75, 80, 95]

average = calculate_student_performance(student_marks)

print("Student Average:", average)

[LOG] Calling calculate_student_performance
[LOG] Result: 85.0
Student Average: 85.0
